## Step 1 — Import Preprocessing Libraries

This step imports the libraries required to prepare the UJIIndoorLoc dataset for machine-learning experiments.

The official training and validation datasets will remain separated. The original validation dataset will be preserved as the final test set.

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

RANDOM_STATE = 42

print("Preprocessing environment ready")

Preprocessing environment ready


## Step 2 — Load the Raw UJIIndoorLoc Dataset

The original training and validation files are loaded directly from the raw-data directory.

The provided validation dataset will be treated as the final test set and will not be used for model training.

In [5]:
train_raw = pd.read_csv("../data/raw/trainingData.csv")
test_raw = pd.read_csv("../data/raw/validationData.csv")

print("Original training shape:", train_raw.shape)
print("Original test shape:", test_raw.shape)

Original training shape: (19937, 529)
Original test shape: (1111, 529)


## Step 3 — Identify Wi-Fi Fingerprint Features

The Wi-Fi fingerprint is represented by columns beginning with `WAP`.

These columns contain Received Signal Strength Indicator (RSSI) measurements for the detected Wi-Fi access points.

In [6]:
wifi_cols = [col for col in train_raw.columns if col.startswith("WAP")]

print("Number of Wi-Fi features:", len(wifi_cols))

Number of Wi-Fi features: 520


## Step 4 — Represent Undetected Access Points

In UJIIndoorLoc, the value `100` represents an access point that was not detected.

This value must not be interpreted as a strong RSSI measurement.

For the machine-learning experiments, undetected access points are represented as `-110 dBm`, slightly below the weakest observed Wi-Fi signals.

This preserves the distinction between detected and undetected access points while maintaining a physically meaningful RSSI scale.

In [7]:
MISSING_RSSI = -110

X_full = train_raw[wifi_cols].copy()
X_test = test_raw[wifi_cols].copy()

X_full = X_full.replace(100, MISSING_RSSI)
X_test = X_test.replace(100, MISSING_RSSI)

print("Training RSSI range:",
      X_full.min().min(),
      "to",
      X_full.max().max())

print("Test RSSI range:",
      X_test.min().min(),
      "to",
      X_test.max().max())

Training RSSI range: -110 to 0
Test RSSI range: -110 to -34


## Step 5 — Create the Building-Floor Classification Target

A combined building-floor label is created.

This provides a more meaningful indoor-localization classification problem than predicting the building alone.

In [8]:
y_class_full = (
    train_raw["BUILDINGID"].astype(str)
    + "_"
    + train_raw["FLOOR"].astype(str)
)

y_class_test = (
    test_raw["BUILDINGID"].astype(str)
    + "_"
    + test_raw["FLOOR"].astype(str)
)

print("Number of building-floor classes:",
      y_class_full.nunique())

print("\nClasses:")
print(sorted(y_class_full.unique()))

Number of building-floor classes: 13

Classes:
['0_0', '0_1', '0_2', '0_3', '1_0', '1_1', '1_2', '1_3', '2_0', '2_1', '2_2', '2_3', '2_4']


## Step 6 — Create the Coordinate Regression Target

Longitude and latitude are retained as continuous target variables for coordinate-based localization evaluation.

The same train, development, and test observations will later be used for both classification and coordinate regression.

In [9]:
y_coord_full = train_raw[["LONGITUDE", "LATITUDE"]].copy()
y_coord_test = test_raw[["LONGITUDE", "LATITUDE"]].copy()

display(y_coord_full.head())

,LONGITUDE,LATITUDE
0,-7541.2643,4.864921e+06
1,-7536.6212,4.864934e+06
2,-7519.1524,4.864950e+06
3,-7524.5704,4.864934e+06
4,-7632.1436,4.864982e+06


## Step 7 — Create Training and Development Sets

The original UJIIndoorLoc training dataset is divided into training and development subsets.

A stratified split based on the building-floor class is used so that the class distribution remains approximately consistent.

The original UJIIndoorLoc validation dataset remains untouched and serves as the final test set.

In [10]:
indices = np.arange(len(X_full))

train_idx, dev_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_class_full
)

X_train = X_full.iloc[train_idx].copy()
X_dev = X_full.iloc[dev_idx].copy()

y_class_train = y_class_full.iloc[train_idx].copy()
y_class_dev = y_class_full.iloc[dev_idx].copy()

y_coord_train = y_coord_full.iloc[train_idx].copy()
y_coord_dev = y_coord_full.iloc[dev_idx].copy()

print("Training samples:", len(X_train))
print("Development samples:", len(X_dev))
print("Final test samples:", len(X_test))

Training samples: 15949
Development samples: 3988
Final test samples: 1111


## Step 8 — Normalize Wi-Fi RSSI Features

Min-max normalization is applied to Wi-Fi RSSI features.

The scaler is fitted using the training subset only to prevent information leakage.

The same fitted transformation is then applied to the development and final test sets.

In [11]:
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_dev_scaled = scaler.transform(X_dev)
X_test_scaled = scaler.transform(X_test)

print("Scaled training range:",
      X_train_scaled.min(),
      "to",
      X_train_scaled.max())

Scaled training range: 0.0 to 1.0000000000000004


## Step 9 — Validate the Preprocessed Dataset

This step checks for missing values, infinite values, and dimensional inconsistencies before machine-learning experiments begin.

In [12]:
print("NaN in training:",
      np.isnan(X_train_scaled).sum())

print("NaN in development:",
      np.isnan(X_dev_scaled).sum())

print("NaN in test:",
      np.isnan(X_test_scaled).sum())

print("Training shape:", X_train_scaled.shape)
print("Development shape:", X_dev_scaled.shape)
print("Test shape:", X_test_scaled.shape)

NaN in training: 0
NaN in development: 0
NaN in test: 0
Training shape: (15949, 520)
Development shape: (3988, 520)
Test shape: (1111, 520)


## Step 10 — Save Preprocessed Experimental Data

The processed arrays and target variables are saved so that all subsequent experiments use exactly the same dataset split and preprocessing configuration.

This improves reproducibility across the baseline and robustness experiments.

In [13]:
np.savez_compressed(
    "../data/processed/uji_preprocessed.npz",

    X_train=X_train_scaled,
    X_dev=X_dev_scaled,
    X_test=X_test_scaled,

    y_class_train=y_class_train.to_numpy(),
    y_class_dev=y_class_dev.to_numpy(),
    y_class_test=y_class_test.to_numpy(),

    y_coord_train=y_coord_train.to_numpy(),
    y_coord_dev=y_coord_dev.to_numpy(),
    y_coord_test=y_coord_test.to_numpy(),

    train_idx=train_idx,
    dev_idx=dev_idx
)

print("Processed dataset saved successfully.")

Processed dataset saved successfully.


In [14]:
np.savez_compressed(
    "../data/processed/uji_rssi_raw_processed.npz",

    X_train=X_train.to_numpy(),
    X_dev=X_dev.to_numpy(),
    X_test=X_test.to_numpy()
)

print("RSSI-domain data saved for robustness experiments.")

RSSI-domain data saved for robustness experiments.


## Step 11 — Preprocessing Summary

This final check documents the experimental dataset configuration that will be used throughout the research.

In [15]:
print("=== DEN702 Experimental Dataset ===")
print("Wi-Fi features:", len(wifi_cols))
print("Training:", X_train_scaled.shape)
print("Development:", X_dev_scaled.shape)
print("Final test:", X_test_scaled.shape)
print("Classification classes:", y_class_full.nunique())
print("Missing AP representation:", MISSING_RSSI, "dBm")
print("Random state:", RANDOM_STATE)

=== DEN702 Experimental Dataset ===
Wi-Fi features: 520
Training: (15949, 520)
Development: (3988, 520)
Final test: (1111, 520)
Classification classes: 13
Missing AP representation: -110 dBm
Random state: 42


In [1]:
# ============================================================
# DEN702 Research Paper
# Notebook 02 — UJIIndoorLoc Preprocessing
#
# Research Topic:
# Robust Wi-Fi Fingerprint-Based Indoor Localization
# Under Access-Point Loss and RSSI Noise
#
# Purpose:
# 1. Load the original UJIIndoorLoc dataset
# 2. Preserve the official validation dataset as final test data
# 3. Represent undetected APs using -110 dBm
# 4. Create building-floor classification targets
# 5. Create coordinate regression targets
# 6. Create training/development split
# 7. Normalize RSSI features without data leakage
# 8. Preserve observed-AP masks for robustness experiments
# 9. Save all processed data for reproducible experiments
# ============================================================


# ------------------------------------------------------------
# Step 1 — Import Required Libraries
#
# These libraries are used for preprocessing, splitting,
# normalization, validation, and saving experimental artifacts.
# ------------------------------------------------------------

from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler


print("=" * 70)
print("DEN702 — UJIIndoorLoc Preprocessing")
print("=" * 70)


# ------------------------------------------------------------
# Step 2 — Define Reproducibility Settings
#
# The same random seed must be used throughout the research
# so that train/development splits and stochastic experiments
# can be reproduced.
# ------------------------------------------------------------

RANDOM_STATE = 42

# UJIIndoorLoc uses 100 to indicate that an AP was not detected.
# For our ML representation, we replace this value with -110 dBm.
#
# -110 dBm was selected because it lies below the weakest
# observed RSSI value (-104 dBm) in Notebook 01.

MISSING_RSSI = -110


print("\nRandom state:", RANDOM_STATE)
print("Missing AP representation:", MISSING_RSSI, "dBm")


# ------------------------------------------------------------
# Step 3 — Detect Project Directory
#
# This allows the notebook to work whether it is launched from
# the notebooks folder or from the project root.
# ------------------------------------------------------------

if Path("../data/raw/trainingData.csv").exists():

    PROJECT_ROOT = Path("..").resolve()

elif Path("data/raw/trainingData.csv").exists():

    PROJECT_ROOT = Path(".").resolve()

else:

    raise FileNotFoundError(
        "Could not locate data/raw/trainingData.csv.\n"
        "Make sure the UJIIndoorLoc CSV files are stored "
        "inside data/raw/."
    )


RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_CSV_DIR = PROJECT_ROOT / "results" / "csv"

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_CSV_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TRAIN_FILE = RAW_DATA_DIR / "trainingData.csv"
TEST_FILE = RAW_DATA_DIR / "validationData.csv"


print("\nProject root:")
print(PROJECT_ROOT)


# ------------------------------------------------------------
# Step 4 — Load Original Dataset
#
# The original trainingData.csv dataset will be divided into
# training and development subsets.
#
# The original validationData.csv dataset is preserved as the
# final independent test set.
# ------------------------------------------------------------

train_raw = pd.read_csv(TRAIN_FILE)
test_raw = pd.read_csv(TEST_FILE)


print("\n" + "=" * 70)
print("STEP 4 — RAW DATASET")
print("=" * 70)

print("Original training shape:", train_raw.shape)
print("Final test shape:", test_raw.shape)


# ------------------------------------------------------------
# Step 5 — Identify Wi-Fi RSSI Features
#
# All columns beginning with WAP represent Wi-Fi access-point
# RSSI measurements.
# ------------------------------------------------------------

wifi_cols = [
    column
    for column in train_raw.columns
    if column.startswith("WAP")
]


print("\n" + "=" * 70)
print("STEP 5 — WI-FI FEATURES")
print("=" * 70)

print("Number of Wi-Fi features:", len(wifi_cols))


if len(wifi_cols) != 520:
    raise ValueError(
        f"Expected 520 WAP features but found {len(wifi_cols)}."
    )


# ------------------------------------------------------------
# Step 6 — Preserve Observed-AP Masks
#
# IMPORTANT FOR THE RESEARCH:
#
# A value of 100 means that the AP was not detected.
#
# We preserve a Boolean mask BEFORE replacing 100.
#
# True  = AP detected
# False = AP not detected
#
# Later, AP-dropout experiments will remove a percentage of
# detected APs only. This prevents us from "dropping" APs that
# were already missing.
# ------------------------------------------------------------

observed_mask_full = (
    train_raw[wifi_cols] != 100
).to_numpy(dtype=bool)

observed_mask_test = (
    test_raw[wifi_cols] != 100
).to_numpy(dtype=bool)


observed_count_full = observed_mask_full.sum(axis=1)
observed_count_test = observed_mask_test.sum(axis=1)


print("\n" + "=" * 70)
print("STEP 6 — OBSERVED ACCESS POINT STATISTICS")
print("=" * 70)

print(
    "Average detected APs per training fingerprint:",
    round(observed_count_full.mean(), 2)
)

print(
    "Median detected APs per training fingerprint:",
    round(float(np.median(observed_count_full)), 2)
)

print(
    "Minimum detected APs:",
    int(observed_count_full.min())
)

print(
    "Maximum detected APs:",
    int(observed_count_full.max())
)


# ------------------------------------------------------------
# Step 7 — Replace Undetected AP Marker
#
# Original representation:
#
#      100 = AP not detected
#
# Experimental representation:
#
#     -110 = AP not detected
#
# This prevents the value 100 from being interpreted as a
# very strong RSSI measurement.
# ------------------------------------------------------------

X_full_raw = (
    train_raw[wifi_cols]
    .replace(100, MISSING_RSSI)
    .copy()
)

X_test_raw = (
    test_raw[wifi_cols]
    .replace(100, MISSING_RSSI)
    .copy()
)


print("\n" + "=" * 70)
print("STEP 7 — RSSI REPRESENTATION")
print("=" * 70)

print(
    "Training RSSI range:",
    X_full_raw.min().min(),
    "to",
    X_full_raw.max().max()
)

print(
    "Final test RSSI range:",
    X_test_raw.min().min(),
    "to",
    X_test_raw.max().max()
)


# Confirm that no original 100 values remain

remaining_100_train = (
    X_full_raw == 100
).sum().sum()

remaining_100_test = (
    X_test_raw == 100
).sum().sum()


print(
    "Remaining value-100 entries in training:",
    remaining_100_train
)

print(
    "Remaining value-100 entries in test:",
    remaining_100_test
)


# ------------------------------------------------------------
# Step 8 — Create Building-Floor Classification Target
#
# Instead of predicting only the building or only the floor,
# we create a combined label:
#
#     BUILDINGID_FLOOR
#
# Example:
#
#     0_0
#     0_1
#     1_2
#
# This produces a more meaningful indoor-location
# classification task.
# ------------------------------------------------------------

y_class_full = (
    train_raw["BUILDINGID"].astype(str)
    + "_"
    + train_raw["FLOOR"].astype(str)
)

y_class_test = (
    test_raw["BUILDINGID"].astype(str)
    + "_"
    + test_raw["FLOOR"].astype(str)
)


classes = sorted(
    y_class_full.unique()
)


print("\n" + "=" * 70)
print("STEP 8 — CLASSIFICATION TARGET")
print("=" * 70)

print(
    "Number of Building-Floor classes:",
    len(classes)
)

print("\nClasses:")
print(classes)


# ------------------------------------------------------------
# Step 9 — Verify Final-Test Classes
#
# The final test dataset should not contain an unseen
# building-floor class.
# ------------------------------------------------------------

train_class_set = set(
    y_class_full.unique()
)

test_class_set = set(
    y_class_test.unique()
)

unseen_test_classes = (
    test_class_set
    - train_class_set
)


print("\nClasses appearing only in final test:")
print(unseen_test_classes)


if unseen_test_classes:

    raise ValueError(
        "The final test dataset contains classes "
        "not present in the training dataset."
    )


# ------------------------------------------------------------
# Step 10 — Create Coordinate Regression Targets
#
# Longitude and latitude are preserved as continuous labels for
# coordinate-localization experiments.
# ------------------------------------------------------------

y_coord_full = (
    train_raw[
        [
            "LONGITUDE",
            "LATITUDE"
        ]
    ]
    .copy()
)

y_coord_test = (
    test_raw[
        [
            "LONGITUDE",
            "LATITUDE"
        ]
    ]
    .copy()
)


print("\n" + "=" * 70)
print("STEP 10 — COORDINATE TARGET")
print("=" * 70)

print("Coordinate target shape:", y_coord_full.shape)

print("\nFirst five coordinate targets:")

display(
    y_coord_full.head()
)


# ------------------------------------------------------------
# Step 11 — Create Train / Development Split
#
# The official training dataset is split:
#
#     80% Training
#     20% Development
#
# Stratification is performed using the building-floor class.
#
# The official validationData.csv remains untouched and is
# reserved as the final independent test dataset.
# ------------------------------------------------------------

all_indices = np.arange(
    len(X_full_raw)
)


train_idx, dev_idx = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_class_full
)


print("\n" + "=" * 70)
print("STEP 11 — DATA SPLIT")
print("=" * 70)

print("Training samples:", len(train_idx))
print("Development samples:", len(dev_idx))
print("Final test samples:", len(X_test_raw))


# ------------------------------------------------------------
# Step 12 — Apply Split to RSSI Features
# ------------------------------------------------------------

X_train_raw = (
    X_full_raw
    .iloc[train_idx]
    .copy()
)

X_dev_raw = (
    X_full_raw
    .iloc[dev_idx]
    .copy()
)


# ------------------------------------------------------------
# Step 13 — Apply Split to Observed-AP Masks
#
# These masks will later allow AP-loss simulations to modify
# only currently detected access points.
# ------------------------------------------------------------

observed_mask_train = (
    observed_mask_full[train_idx]
)

observed_mask_dev = (
    observed_mask_full[dev_idx]
)


print("\nObserved mask shapes:")

print(
    "Training:",
    observed_mask_train.shape
)

print(
    "Development:",
    observed_mask_dev.shape
)

print(
    "Final test:",
    observed_mask_test.shape
)


# ------------------------------------------------------------
# Step 14 — Apply Split to Classification Targets
# ------------------------------------------------------------

y_class_train = (
    y_class_full
    .iloc[train_idx]
    .copy()
)

y_class_dev = (
    y_class_full
    .iloc[dev_idx]
    .copy()
)


# ------------------------------------------------------------
# Step 15 — Apply Split to Coordinate Targets
# ------------------------------------------------------------

y_coord_train = (
    y_coord_full
    .iloc[train_idx]
    .copy()
)

y_coord_dev = (
    y_coord_full
    .iloc[dev_idx]
    .copy()
)


# ------------------------------------------------------------
# Step 16 — Fit RSSI Normalization
#
# MinMaxScaler is fitted ONLY using training data.
#
# This prevents development/test information from leaking
# into preprocessing.
# ------------------------------------------------------------

scaler = MinMaxScaler()


X_train_scaled = scaler.fit_transform(
    X_train_raw
)

X_dev_scaled = scaler.transform(
    X_dev_raw
)

X_test_scaled = scaler.transform(
    X_test_raw
)


print("\n" + "=" * 70)
print("STEP 16 — FEATURE NORMALIZATION")
print("=" * 70)

print(
    "Scaled training minimum:",
    X_train_scaled.min()
)

print(
    "Scaled training maximum:",
    X_train_scaled.max()
)


# ------------------------------------------------------------
# Step 17 — Validate Processed Data
#
# Check for NaN values and infinities before training models.
# ------------------------------------------------------------

def validate_array(name, array):

    nan_count = np.isnan(array).sum()
    inf_count = np.isinf(array).sum()

    print(
        f"{name}: "
        f"shape={array.shape}, "
        f"NaN={nan_count}, "
        f"Inf={inf_count}"
    )

    if nan_count != 0:
        raise ValueError(
            f"{name} contains NaN values."
        )

    if inf_count != 0:
        raise ValueError(
            f"{name} contains infinite values."
        )


print("\n" + "=" * 70)
print("STEP 17 — DATA VALIDATION")
print("=" * 70)


validate_array(
    "X_train_scaled",
    X_train_scaled
)

validate_array(
    "X_dev_scaled",
    X_dev_scaled
)

validate_array(
    "X_test_scaled",
    X_test_scaled
)


# ------------------------------------------------------------
# Step 18 — Verify Split Class Distribution
#
# This checks that stratification preserved all classes in both
# the training and development datasets.
# ------------------------------------------------------------

train_classes = set(
    y_class_train.unique()
)

dev_classes = set(
    y_class_dev.unique()
)


print("\nTraining classes:", len(train_classes))
print("Development classes:", len(dev_classes))

print(
    "Classes missing from development:",
    train_classes - dev_classes
)


if train_classes != dev_classes:

    print(
        "\nWARNING: Training and development "
        "class sets are not identical."
    )


# ------------------------------------------------------------
# Step 19 — Save Scaler
#
# The exact scaler used here must also be used during the
# AP-loss and RSSI-noise robustness experiments.
# ------------------------------------------------------------

SCALER_FILE = (
    PROCESSED_DATA_DIR
    / "uji_minmax_scaler.joblib"
)


joblib.dump(
    scaler,
    SCALER_FILE
)


print("\nScaler saved to:")
print(SCALER_FILE)


# ------------------------------------------------------------
# Step 20 — Save Complete Experimental Dataset
#
# We save:
#
# - scaled RSSI arrays
# - unscaled RSSI arrays
# - observed-AP masks
# - classification targets
# - coordinate targets
# - split indices
# - Wi-Fi feature names
#
# This allows later notebooks to reproduce experiments without
# rebuilding preprocessing each time.
# ------------------------------------------------------------

PROCESSED_FILE = (
    PROCESSED_DATA_DIR
    / "uji_preprocessed.npz"
)


np.savez_compressed(

    PROCESSED_FILE,

    # Scaled model input
    X_train=X_train_scaled,
    X_dev=X_dev_scaled,
    X_test=X_test_scaled,

    # RSSI values in original dBm domain
    X_train_raw=X_train_raw.to_numpy(),
    X_dev_raw=X_dev_raw.to_numpy(),
    X_test_raw=X_test_raw.to_numpy(),

    # Observed AP masks
    observed_mask_train=observed_mask_train,
    observed_mask_dev=observed_mask_dev,
    observed_mask_test=observed_mask_test,

    # Classification targets
    y_class_train=y_class_train.to_numpy(dtype=str),
    y_class_dev=y_class_dev.to_numpy(dtype=str),
    y_class_test=y_class_test.to_numpy(dtype=str),

    # Coordinate regression targets
    y_coord_train=y_coord_train.to_numpy(),
    y_coord_dev=y_coord_dev.to_numpy(),
    y_coord_test=y_coord_test.to_numpy(),

    # Split information
    train_idx=train_idx,
    dev_idx=dev_idx,

    # Wi-Fi columns
    wifi_cols=np.array(
        wifi_cols,
        dtype=str
    ),

    # Research configuration
    missing_rssi=np.array(
        [MISSING_RSSI]
    ),

    random_state=np.array(
        [RANDOM_STATE]
    )
)


print("\nProcessed experimental dataset saved to:")
print(PROCESSED_FILE)


# ------------------------------------------------------------
# Step 21 — Save Wi-Fi Feature List
# ------------------------------------------------------------

wifi_columns_df = pd.DataFrame(
    {
        "Feature": wifi_cols
    }
)


WIFI_COLUMNS_FILE = (
    PROCESSED_DATA_DIR
    / "wifi_columns.csv"
)


wifi_columns_df.to_csv(
    WIFI_COLUMNS_FILE,
    index=False
)


print("\nWi-Fi feature list saved to:")
print(WIFI_COLUMNS_FILE)


# ------------------------------------------------------------
# Step 22 — Create Preprocessing Summary
#
# This summary records important methodological information
# that will later be used in the paper.
# ------------------------------------------------------------

preprocessing_summary = {

    "Original Training Samples":
        len(train_raw),

    "Training Samples":
        len(train_idx),

    "Development Samples":
        len(dev_idx),

    "Final Test Samples":
        len(test_raw),

    "WiFi Features":
        len(wifi_cols),

    "Building-Floor Classes":
        len(classes),

    "Missing AP Original Marker":
        100,

    "Missing AP Experimental Representation":
        MISSING_RSSI,

    "Average Detected APs Per Training Fingerprint":
        float(observed_count_full.mean()),

    "Median Detected APs Per Training Fingerprint":
        float(np.median(observed_count_full)),

    "Minimum Detected APs Per Training Fingerprint":
        int(observed_count_full.min()),

    "Maximum Detected APs Per Training Fingerprint":
        int(observed_count_full.max()),

    "Train Split Percentage":
        80,

    "Development Split Percentage":
        20,

    "Random State":
        RANDOM_STATE,

    "Scaler":
        "MinMaxScaler",

    "Scaler Fit Data":
        "Training subset only"
}


summary_df = pd.DataFrame(
    preprocessing_summary.items(),
    columns=[
        "Setting",
        "Value"
    ]
)


PREPROCESSING_SUMMARY_FILE = (
    RESULTS_CSV_DIR
    / "02_preprocessing_summary.csv"
)


summary_df.to_csv(
    PREPROCESSING_SUMMARY_FILE,
    index=False
)


print("\n" + "=" * 70)
print("STEP 22 — PREPROCESSING SUMMARY")
print("=" * 70)

display(summary_df)


print("\nPreprocessing summary saved to:")
print(PREPROCESSING_SUMMARY_FILE)


# ------------------------------------------------------------
# Step 23 — Final Verification
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("NOTEBOOK 02 COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    "Wi-Fi features:",
    len(wifi_cols)
)

print(
    "Training:",
    X_train_scaled.shape
)

print(
    "Development:",
    X_dev_scaled.shape
)

print(
    "Final test:",
    X_test_scaled.shape
)

print(
    "Classification classes:",
    len(classes)
)

print(
    "Missing AP representation:",
    MISSING_RSSI,
    "dBm"
)

print(
    "Average observed APs per fingerprint:",
    round(
        observed_count_full.mean(),
        2
    )
)

print(
    "Median observed APs per fingerprint:",
    round(
        float(
            np.median(
                observed_count_full
            )
        ),
        2
    )
)

print(
    "Random state:",
    RANDOM_STATE
)

print("\nSaved files:")

print(
    "1.",
    PROCESSED_FILE
)

print(
    "2.",
    SCALER_FILE
)

print(
    "3.",
    WIFI_COLUMNS_FILE
)

print(
    "4.",
    PREPROCESSING_SUMMARY_FILE
)

print("\nNotebook 02 is complete.")
print("Next notebook: 03_clean_baselines.ipynb")

DEN702 — UJIIndoorLoc Preprocessing

Random state: 42
Missing AP representation: -110 dBm

Project root:
/Users/am/Documents/ADU/wifi-localization-research

STEP 4 — RAW DATASET
Original training shape: (19937, 529)
Final test shape: (1111, 529)

STEP 5 — WI-FI FEATURES
Number of Wi-Fi features: 520

STEP 6 — OBSERVED ACCESS POINT STATISTICS
Average detected APs per training fingerprint: 17.99
Median detected APs per training fingerprint: 17.0
Minimum detected APs: 0
Maximum detected APs: 51

STEP 7 — RSSI REPRESENTATION
Training RSSI range: -110 to 0
Final test RSSI range: -110 to -34
Remaining value-100 entries in training: 0
Remaining value-100 entries in test: 0

STEP 8 — CLASSIFICATION TARGET
Number of Building-Floor classes: 13

Classes:
['0_0', '0_1', '0_2', '0_3', '1_0', '1_1', '1_2', '1_3', '2_0', '2_1', '2_2', '2_3', '2_4']

Classes appearing only in final test:
set()

STEP 10 — COORDINATE TARGET
Coordinate target shape: (19937, 2)

First five coordinate targets:


,LONGITUDE,LATITUDE
0,-7541.2643,4.864921e+06
1,-7536.6212,4.864934e+06
2,-7519.1524,4.864950e+06
3,-7524.5704,4.864934e+06
4,-7632.1436,4.864982e+06



STEP 11 — DATA SPLIT
Training samples: 15949
Development samples: 3988
Final test samples: 1111

Observed mask shapes:
Training: (15949, 520)
Development: (3988, 520)
Final test: (1111, 520)

STEP 16 — FEATURE NORMALIZATION
Scaled training minimum: 0.0
Scaled training maximum: 1.0000000000000004

STEP 17 — DATA VALIDATION
X_train_scaled: shape=(15949, 520), NaN=0, Inf=0
X_dev_scaled: shape=(3988, 520), NaN=0, Inf=0
X_test_scaled: shape=(1111, 520), NaN=0, Inf=0

Training classes: 13
Development classes: 13
Classes missing from development: set()

Scaler saved to:
/Users/am/Documents/ADU/wifi-localization-research/data/processed/uji_minmax_scaler.joblib

Processed experimental dataset saved to:
/Users/am/Documents/ADU/wifi-localization-research/data/processed/uji_preprocessed.npz

Wi-Fi feature list saved to:
/Users/am/Documents/ADU/wifi-localization-research/data/processed/wifi_columns.csv

STEP 22 — PREPROCESSING SUMMARY


,Setting,Value
0,Original Training Samples,19937
1,Training Samples,15949
2,Development Samples,3988
3,Final Test Samples,1111
4,WiFi Features,520
5,Building-Floor Classes,13
6,Missing AP Original Marker,100
7,Missing AP Experimental Representation,-110
8,Average Detected APs Per Training Fingerprint,17.994834
9,Median Detected APs Per Training Fingerprint,17.0



Preprocessing summary saved to:
/Users/am/Documents/ADU/wifi-localization-research/results/csv/02_preprocessing_summary.csv

NOTEBOOK 02 COMPLETED SUCCESSFULLY
Wi-Fi features: 520
Training: (15949, 520)
Development: (3988, 520)
Final test: (1111, 520)
Classification classes: 13
Missing AP representation: -110 dBm
Average observed APs per fingerprint: 17.99
Median observed APs per fingerprint: 17.0
Random state: 42

Saved files:
1. /Users/am/Documents/ADU/wifi-localization-research/data/processed/uji_preprocessed.npz
2. /Users/am/Documents/ADU/wifi-localization-research/data/processed/uji_minmax_scaler.joblib
3. /Users/am/Documents/ADU/wifi-localization-research/data/processed/wifi_columns.csv
4. /Users/am/Documents/ADU/wifi-localization-research/results/csv/02_preprocessing_summary.csv

Notebook 02 is complete.
Next notebook: 03_clean_baselines.ipynb
